# 02 — Matrix Operations
## Addition, Scalar Multiply, Transpose, Broadcasting

> **Prerequisites:** Notebook 01 (scalars, vectors, matrices)  
> **Difficulty:** ⭐⭐☆☆☆ Beginner-Intermediate  
> **Running example:** House price prediction

---

## What are we learning?

Now that you know what matrices are, let's learn the basic **operations** you can do with them.

Think of it like arithmetic — you learned what numbers are (1, 2, 3...) before you learned to add, subtract, and multiply them. Same idea here.

| Operation | What it does | ML use |
|---|---|---|
| **Addition** | Combine two same-shape arrays | Adding bias to layer output |
| **Scalar multiply** | Scale every element by one number | Learning rate × gradient |
| **Transpose** | Flip rows ↔ columns | Fix shape mismatches |
| **Element-wise multiply** | Multiply cell-by-cell | Dropout masks, gating |
| **Broadcasting** | Math on different-shaped arrays | Normalizing all rows at once |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Our running dataset: 5 houses × 4 features [size, bedrooms, age, distance]
X = np.array([
    [1500., 3., 10., 5.2],
    [2100., 4.,  5., 2.8],
    [1200., 2., 20., 8.0],
    [1800., 3.,  8., 4.5],
    [2500., 5.,  2., 1.0],
])

print("Dataset X (5 houses × 4 features):")
print(X)
print(f"Shape: {X.shape}")

---
## 1. Matrix Addition

### Plain English
**Add two arrays by adding the number at each matching position.**

Imagine two spreadsheets with the same columns and rows — you add them by combining each cell with the matching cell in the other spreadsheet.

```
[1, 2]   +   [5, 6]   =   [6,  8]
[3, 4]       [7, 8]       [10, 12]
```

### The ONE rule
> Both arrays must have **exactly the same shape**. Period.

You cannot add a (5, 4) matrix to a (3, 4) matrix. The shapes don't match.

### In ML, addition is used for:
- **Bias term**: every layer output `z = X @ W + b` adds a bias vector
- **Residual connections** in ResNets: `output = layer(x) + x` (add the skip connection)
- **Gradient accumulation**: add up gradients from multiple mini-batches
- **Ensembles**: average predictions from multiple models = sum + divide

In [ ]:
# ─────────────────────────────────────────────────────────
# MATRIX ADDITION: combine two same-shape arrays
# ─────────────────────────────────────────────────────────

# Think of A as "this month's house data" and B as "price adjustments"
A = np.array([[1000., 2., 15.],
              [1500., 3., 10.],
              [2000., 4.,  5.]])

B = np.array([[100., 0., -2.],    # house 1 adjustment: +100 sqft, same beds, -2 yrs
              [0.,   1.,  0.],    # house 2: same sqft, +1 bedroom, same age
              [-50., 0.,  1.]])   # house 3: -50 sqft, same beds, +1 year old

print("A:")
print(A)
print("\nB (adjustments):")
print(B)
print("\nA + B (updated values):")
print(A + B)  # adds element-by-element
print("\nA - B (reverse adjustments):")
print(A - B)

print()
print("What happens if shapes don't match?")
try:
    bad_result = A + np.array([[1, 2, 3, 4]])  # (3,3) + (1,4) — doesn't work
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# ─────────────────────────────────────────────────────────
# ML EXAMPLE: adding bias to a neural network layer
# ─────────────────────────────────────────────────────────

# Imagine 3 houses going through a neural network layer
# W transforms 4 features → 3 neurons
np.random.seed(42)
X_batch = np.array([[1500., 3., 10., 5.2],   # house 1 (features)
                    [2100., 4.,  5., 2.8],   # house 2
                    [1200., 2., 20., 8.0]])  # house 3

W = np.random.randn(4, 3) * 0.01   # weight matrix (4 inputs → 3 neurons)
bias = np.array([0.1, -0.2, 0.3])  # one bias per neuron (3 values)

# Forward pass: output = X @ W + bias
raw_output = X_batch @ W            # shape: (3 houses, 3 neurons)
print("Raw output (before bias), shape:", raw_output.shape)
print(raw_output.round(3))

# Adding bias: each row gets the SAME bias added
# raw_output is (3, 3), bias is (3,) → NumPy broadcasts (explained in section 5)
z = raw_output + bias   # bias added to every row
print("\nAfter adding bias, shape:", z.shape)
print(z.round(3))
print("\n→ The bias term shifts the activation of each neuron up or down")

---
## 2. Scalar Multiplication

### Plain English
**Multiply every element of a matrix by one number (the scalar).**

Like adjusting the brightness of every pixel in a photo by the same factor.

```
3 × [[1, 2],  =  [[3,  6],
      [3, 4]]      [9, 12]]
```

### In ML, scalar multiplication is used for:
- **Learning rate**: `new_weights = old_weights - 0.001 × gradients`
  - The `0.001` is scalar multiplication — scales down the gradient
- **Normalizing**: dividing by the max value, or by the norm
- **Temperature scaling**: dividing logits by a temperature T before softmax
- **Weight decay**: multiplying weights by a small value to gradually shrink them

In [ ]:
# ─────────────────────────────────────────────────────────
# SCALAR MULTIPLICATION
# ─────────────────────────────────────────────────────────

prices = np.array([250000., 320000., 190000., 280000., 410000.])

print("=== Basic scalar multiplication ===")
print(f"Original prices: {prices}")
print(f"After 10% price increase (× 1.10): {prices * 1.10}")
print(f"Converted to $000s (÷ 1000):       {prices / 1000}")
print(f"Flipped signs (× -1):              {prices * -1}")

print()
print("=== ML example: gradient descent weight update ===")
# Gradient descent: new_weights = old_weights - learning_rate × gradient
weights = np.array([150., 20000., -500., -8000.])   # model weights
gradient = np.array([25., 3000., -80., -1200.])      # how to improve
learning_rate = 0.001                                # step size (scalar)

step = learning_rate * gradient    # scale the gradient down
new_weights = weights - step

print(f"Old weights: {weights}")
print(f"Gradient:    {gradient}")
print(f"LR × grad:   {step}")
print(f"New weights: {new_weights}")
print(f"\nDifference (change): {new_weights - weights}")
print("→ Weights moved just a tiny bit in the direction that reduces error")

---
## 3. Matrix Transposition

### Plain English
**Flip the matrix so that rows become columns and columns become rows.**

```
A = [[1, 2, 3],    Aᵀ = [[1, 4],
     [4, 5, 6]]          [2, 5],
                          [3, 6]]
```

If A has shape **(m × n)**, then Aᵀ ("A transpose") has shape **(n × m)**.

### The rule
> Element at row i, column j of A  →  goes to row j, column i in Aᵀ

So Aᵀ[i, j] = A[j, i]

### In ML, transposition is used for:
- **Matrix multiplication shape-fixing**: if A is (m×n) and B is (m×p), you can't multiply them directly. But Aᵀ is (n×m) and can multiply B (m×p) → gives (n×p)
- **Gram matrix**: XᵀX gives feature correlations — shape (features × features)
- **Backpropagation**: gradients flow through the transpose of weight matrices
- **Symmetric check**: A matrix is symmetric if A = Aᵀ (e.g., covariance matrices)

In [ ]:
# ─────────────────────────────────────────────────────────
# TRANSPOSITION
# ─────────────────────────────────────────────────────────

A = np.array([[1., 2., 3.],
              [4., 5., 6.]])

print("=== Transpose example ===")
print(f"A, shape {A.shape}:")
print(A)
print()
print(f"A.T, shape {A.T.shape}:")
print(A.T)

print()
print("=== Verify: A.T[i,j] == A[j,i] ===")
print(f"A[0, 2] = {A[0, 2]}  →  A.T[2, 0] = {A.T[2, 0]}  (same!)")
print(f"A[1, 1] = {A[1, 1]}  →  A.T[1, 1] = {A.T[1, 1]}  (diagonal stays)")

print()
print("=== The 1D transpose trap ===")
v = np.array([1., 2., 3., 4.])  # shape (4,)
print(f"v       = {v}, shape = {v.shape}")
print(f"v.T     = {v.T}, shape = {v.T.shape}   ← SAME! .T doesn't change 1D arrays")
print()
print("FIX: use reshape to make it a proper 2D column vector:")
v_col = v.reshape(-1, 1)  # shape (4, 1)
print(f"v.reshape(-1,1), shape = {v_col.shape}:")
print(v_col)

In [ ]:
# ─────────────────────────────────────────────────────────
# ML EXAMPLE: Gram matrix and symmetric matrices
# ─────────────────────────────────────────────────────────

print("=== Gram matrix: XᵀX ===")
print("The dataset matrix:")
print(X)
print(f"Shape: {X.shape}  (5 houses × 4 features)")

# Gram matrix: XᵀX captures how features correlate with each other
Gram = X.T @ X     # (4×5) @ (5×4) = (4×4)
print(f"\nXᵀX (Gram matrix), shape {Gram.shape}:")
print(Gram.round(0))
print("→ This is a 4×4 matrix: one value for each pair of features")
print("→ Diagonal: how much each feature 'points in a similar direction' to itself")
print("→ Off-diagonal: how correlated two features are")

# Symmetric check
print(f"\nIs XᵀX symmetric (XᵀX == (XᵀX)ᵀ)? {np.allclose(Gram, Gram.T)}")
print("→ Yes! XᵀX is always symmetric. This is an important property.")

---
## 4. Element-wise (Hadamard) Product

### Plain English
**Multiply two same-shape arrays by multiplying each cell with the cell in the same position.**

```
[[1, 2],  ⊙  [[5, 6],  =  [[1×5, 2×6],  =  [[5, 12],
 [3, 4]]      [7, 8]]      [3×7, 4×8]]       [21, 32]]
```

### ⚠️ This is NOT matrix multiplication!
- Element-wise: `A * B` (the `*` operator in NumPy) — same shape required
- Matrix multiply: `A @ B` (the `@` operator) — different rule (covered in Notebook 03)

### In ML, element-wise multiply is used for:
- **Dropout**: randomly zero out neurons by multiplying with a 0/1 mask
- **LSTM gates**: forget gate × cell state keeps or erases memory
- **Attention**: query-key similarity scores weight the values

In [ ]:
# ─────────────────────────────────────────────────────────
# ELEMENT-WISE PRODUCT vs MATRIX MULTIPLY
# ─────────────────────────────────────────────────────────

A = np.array([[1., 2.],
              [3., 4.]])
B = np.array([[5., 6.],
              [7., 8.]])

print("A:")
print(A)
print("B:")
print(B)

print()
print("Element-wise: A * B  (same position × same position)")
print(A * B)   # [[1*5, 2*6], [3*7, 4*8]]

print()
print("Matrix multiply: A @ B  (rows of A dotted with columns of B)")
print(A @ B)   # [[1*5+2*7, 1*6+2*8], [3*5+4*7, 3*6+4*8]]
print()
print("These are COMPLETELY different! Don't confuse * and @")

print()
print("=== ML example: Dropout ===")
# Dropout: randomly set some neuron outputs to 0
np.random.seed(42)
activations = np.array([[0.8, 0.3, 0.9, 0.1, 0.7]])   # neuron activations

# Create a random mask: 1 = keep, 0 = drop
dropout_rate = 0.4
mask = (np.random.rand(*activations.shape) > dropout_rate).astype(float)
print(f"Activations: {activations}")
print(f"Dropout mask (1=keep, 0=drop): {mask}")
print(f"After dropout (A * mask): {activations * mask}")
print(f"→ {int(mask.sum())} neurons kept, {int((1-mask).sum())} dropped")

---
## 5. Broadcasting — Operations on Different Shapes

### Plain English
**NumPy can do math between arrays of different sizes by automatically "stretching" the smaller array.**

This sounds abstract, but you use it constantly without thinking about it.

### Common example: normalize a dataset
```python
X_norm = (X - X.mean(axis=0)) / X.std(axis=0)
```
Here X is (5, 4) but `X.mean(axis=0)` is (4,). NumPy stretches the mean across all 5 rows.

### Broadcasting rules (simplified)
1. Arrays are compared dimension by dimension, **from right to left**
2. Dimensions are compatible if they are **equal** OR **one of them is 1**
3. The dimension of size 1 gets "stretched" to match the other

### Why it matters in ML
- **Normalizing all samples**: subtract mean, divide by std — without looping
- **Adding bias**: `(n_samples, n_neurons) + (n_neurons,)` — bias added to every row
- **Computing distances**: pairwise operations between samples

In [ ]:
# ─────────────────────────────────────────────────────────
# BROADCASTING: the automatic stretching
# ─────────────────────────────────────────────────────────

print("=== Example 1: add a scalar to every element ===")
A = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print(f"A + 10 (scalar broadcasts across ALL {A.size} elements):")
print(A + 10)

print()
print("=== Example 2: add a row vector to every row ===")
# This is how we add bias in neural networks!
B = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])     # shape (3, 3) — 3 samples, 3 neurons
bias = np.array([10., 20., 30.]) # shape (3,) — one bias per neuron
print(f"B shape: {B.shape}, bias shape: {bias.shape}")
print(f"B + bias:")
print(B + bias)   # bias is added to EVERY row of B
print("→ Row 0 gets [1+10, 2+20, 3+30], row 1 gets [4+10, 5+20, 6+30], etc.")

print()
print("=== Example 3: normalize our house dataset ===")
print("Original X:")
print(X)
col_means = X.mean(axis=0)   # mean of each column — shape (4,)
col_stds  = X.std(axis=0)    # std of each column  — shape (4,)
print(f"Column means: {col_means}")
print(f"Column stds:  {col_stds.round(1)}")

X_normalized = (X - col_means) / col_stds   # both col_means and col_stds broadcast!
print(f"\nNormalized X (mean≈0, std≈1 per column):")
print(X_normalized.round(3))
print(f"Normalized column means: {X_normalized.mean(axis=0).round(6)} (should be ~0)")
print(f"Normalized column stds:  {X_normalized.std(axis=0).round(3)} (should be ~1)")

In [ ]:
# ─────────────────────────────────────────────────────────
# VISUALIZING BROADCASTING (normalize before/after)
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

feature_names = ['Size (sqft)', 'Bedrooms', 'Age (yrs)', 'Distance (km)']
colors = ['steelblue', 'orange', 'green', 'red']

# Before normalization: each feature at very different scales
ax1 = axes[0]
for j, (name, color) in enumerate(zip(feature_names, colors)):
    ax1.scatter([j]*5, X[:, j], color=color, s=100, label=name, zorder=3)
    ax1.plot([j-0.1, j+0.1], [X[:, j].mean(), X[:, j].mean()],
             color=color, lw=3)
ax1.set_xticks(range(4)); ax1.set_xticklabels(feature_names, rotation=15)
ax1.set_ylabel('Feature Value (raw scale)')
ax1.set_title('Before Normalization\n(features are on very different scales)', fontsize=10)
ax1.grid(True, alpha=0.3)

# After normalization: all features on same scale
ax2 = axes[1]
for j, (name, color) in enumerate(zip(feature_names, colors)):
    ax2.scatter([j]*5, X_normalized[:, j], color=color, s=100, label=name, zorder=3)
    ax2.plot([j-0.1, j+0.1], [0, 0], color=color, lw=3)
ax2.axhline(0, color='black', lw=1, linestyle='--', alpha=0.5)
ax2.set_xticks(range(4)); ax2.set_xticklabels(feature_names, rotation=15)
ax2.set_ylabel('Normalized Value (z-score)')
ax2.set_title('After Broadcasting Normalization\n(all features centered around 0)', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Broadcasting in Action: Normalizing All Columns at Once', fontsize=12)
plt.tight_layout()
plt.show()
print("Normalization is crucial: without it, 'size' (1500) dominates 'bedrooms' (3)")

---
## Summary

| Operation | Python/NumPy | Rule | ML use |
|---|---|---|---|
| Addition | `A + B` | Same shape required | Bias addition, residuals |
| Subtraction | `A - B` | Same shape required | Error = y_true - y_pred |
| Scalar multiply | `c * A` | Any shape | Learning rate × gradient |
| Element-wise | `A * B` | Same shape required | Dropout, LSTM gates |
| Transpose | `A.T` | Any shape | Shape-fixing, Gram matrix |
| Broadcasting | auto | Compatible dims | Normalization, bias add |

### The most common confusion
```python
# Element-wise multiply (same shape, cell × cell)
result = A * B    # * operator

# Matrix multiply (covered in Notebook 03)
result = A @ B    # @ operator — very different!
```

**Next: Notebook 03 — Matrix Multiplication** (the heart of all ML computations)